In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


import copy

import gymnasium as gym
import matplotlib.pyplot as plt

from torch.utils.tensorboard import SummaryWriter

# import sys, os
# sys.path.append(os.getcwd()+"/../src")
from mcts_torch import MCTSNode, MCTSTree

In [ ]:
def transform_value_reward(x:torch.Tensor, eps = 0.001):
    return torch.sign(x) * (torch.sqrt(torch.abs(x) +1) - 1 + eps*x)

def transform_value_reward_inv(y:torch.Tensor, eps=0.001):
    root = torch.sqrt(y/eps + 1 + torch.sign(y)/eps + (1/(2*eps))**2)
    return torch.sign(y)*((root-1/(2*eps))**2 - 1)

In [ ]:
class DummyRepresentationModel(nn.Module): # s0 = h(past_observations)
    def __init__(self, dim_obs_in, n_hidden, n_hidden_state): # TODO encode several past observations, TODO encode actions
        super().__init__()
        
        self.dim_obs_in = dim_obs_in

        self.hidden_state_net = nn.Sequential(
            nn.Linear(dim_obs_in, n_hidden),
            nn.Tanh(),
            nn.Linear(n_hidden, n_hidden_state),
        )

    def forward(self, obs): # TODO encode action in input to dynamics function
        if not isinstance(obs, torch.Tensor): obs = torch.tensor(obs)
        hidden_state = self.hidden_state_net(obs)
        return hidden_state

class DummyPredictionModel(nn.Module): # p_k, v_k = f(hidden)
    def __init__(self, n_hidden_state, n_actions):
        super().__init__()

        # self.prediction_net_body = nn.Sequential(
        #     nn.
        # )

        self.prediction_net_policy_head = nn.Sequential(
            nn.Tanh(),
            nn.Linear(in_features=n_hidden_state, out_features=n_hidden_state),
            nn.Tanh(),
            nn.Linear(in_features=n_hidden_state, out_features=n_actions)
        )

        self.prediction_net_value_head = nn.Sequential(
            nn.Tanh(),
            nn.Linear(in_features=n_hidden_state, out_features=n_hidden_state),
            nn.Tanh(),
            nn.Linear(in_features=n_hidden_state, out_features=1)
        )

    def forward(self, hidden_state):
        if not isinstance(hidden_state, torch.Tensor): hidden_state = torch.tensor(hidden_state)

        #_prediction_body   = self.prediction_net_body(hidden_state)
        _prediction_body = hidden_state
        policy_logits_pred = self.prediction_net_policy_head(_prediction_body)
        value_logits_pred  = self.prediction_net_value_head(_prediction_body)
        return policy_logits_pred, 50.0*value_logits_pred # transform_value_reward_inv(value_logits_pred)

class DummyDynamicsModel(nn.Module): # r_k, s_k = g(skm1, a_k) # reward, new hidden state
    def __init__(self, n_hidden_state, n_actions):
        super().__init__()
        self.n_actions = n_actions
        # rk, sk = g(skm1, ak) # reward, new hidden state
        # TODO encode action
        self.dynamics_net_body = nn.Sequential(
            nn.Tanh(),
            nn.Linear(in_features=n_hidden_state+1, out_features=n_hidden_state) # down-sampling
        )

        self.dynamics_net_reward_head = nn.Sequential(
            nn.Tanh(),
            nn.Linear(in_features=n_hidden_state, out_features=1)
        )

        self.dynamics_net_next_state_head = nn.Sequential(
            nn.Tanh(),
            nn.Linear(in_features=n_hidden_state, out_features=n_hidden_state),
        )

    def forward(self, hidden_state, action): 
        # discrete action -> TODO transform to [0, 1] by dividing by n_actions (or 1+n_actions)
        # TODO accept action:int input
        if not isinstance(hidden_state, torch.Tensor): hidden_state = torch.tensor(hidden_state)
        if not isinstance(action, torch.Tensor): action = torch.tensor(action)
        if hidden_state.ndim == action.ndim+1: action = action.unsqueeze(action.ndim)
        hidden_state = torch.cat([hidden_state, action], dim=hidden_state.ndim-1)
        _dynamics_body   = self.dynamics_net_body(hidden_state)
        reward_logits_pred = self.dynamics_net_reward_head(_dynamics_body)
        next_state_pred  = self.dynamics_net_next_state_head(_dynamics_body)
        return reward_logits_pred, next_state_pred # TODO maybe reward head is not complex enough to predict 1 most of the time, but 0 sometimes

In [3]:
# # without batch dim
# s = torch.rand(6)
# a = torch.tensor([3 / 10])

# # want shape 7
# print(s)
# print(torch.cat([s, a], dim=s.ndim-1))

# # with batch
# s_batch = torch.rand((2, 6))
# a_batch = torch.tensor([[3 / 10], [4/10]])
# print(a_batch.shape)

# print(s_batch)
# print(torch.cat([s_batch, a_batch], dim=s_batch.ndim-1))


In [4]:
# n_hidden_state = 6
# n_actions=8
# h = RepresentationModel(dim_obs_in=8, n_hidden=16, n_hidden_state=n_hidden_state)
# f = PredictionModel(n_hidden_state, n_actions=n_actions)
# g = DynamicsModel(n_hidden_state=n_hidden_state, n_actions=n_actions)

# o = torch.rand((10, 8))
# h(o).shape

# s = torch.rand((10, 6))
# f(s)[0].shape, f(s)[1].shape

# a_batch = torch.multinomial(torch.ones(10), num_samples=10, replacement=True).reshape((10,1))
# g(s, a_batch)[0].shape, g(s, a_batch)[1].shape

In [5]:
# TODO weighted integer representatation of reward and value)
# TODO scaling of targets with invertible transformation h (page 14)
# Reward loss (cross entropy)
def l_r(input, target):
    return torch.mean(torch.square(input - target), dim=1)

# Value loss (cross entropy)
def l_v(input, target):
    return torch.mean(torch.square(input - target), dim=1)

# Policy loss (cross entropy)
def l_p(input_logits, target_probs):
    """Policy loss function"""
    return nn.CrossEntropyLoss(reduction='none')(input_logits, target_probs)

# Test 
# input_logits = torch.randn((4, 6))
# target_logits = torch.randn((4, 6))
# print(l_p(input_logits, F.softmax(target_logits, dim=1)))

# # This is what I want: pi^T \cdot log p_vector , pi is the target
# # however, it loses a dimension, is not [4, 1], but [4]
# print(- (F.softmax(target_logits, dim=1) * (F.log_softmax(input_logits, dim=1))).sum(dim=1))

# l_r(input_logits, target_logits).shape

In [ ]:
# TODO study cross entropy loss for scalars reward and value
def int_to_support(x):
    l = int(x)
    u = l + 1
    wl = u - x
    wu = x - l
    # x = wl * l + wu * u
    return l, u, wl, wu

x = 1.2 # true value 
y = 1.3 # approximated value
xl, xu, xwl, xwu = int_to_support(x)
yl, yu, ywl, ywu = int_to_support(y)

print(l_p(torch.special.logit(torch.tensor([ywl,ywu])), torch.tensor([xwl, xwu])))
print(l_r(torch.tensor([[ywl, ywu]]), torch.tensor([[xwl, xwu]]))) # TODO fix array dims

tensor(0.5075)
tensor([0.0100])


In [ ]:
# Setup

env_name = "CartPole-v1"
env = gym.make(env_name)


gamma = 0.999
alpha = 0.01 # TODO this is not given in the paper -> look up in pseudocode
buffer_size = 10000
mini_batch_size = 30
n_train_iterations = 10 # how many mini batches are sampled from the buffer
n_MCTS_sims_data_generation = 10
use_MCTS_in_evaluation = True
n_MCTS_sims_evaluation = 10
c = 0.01 # L2 penalty (used in Adam weight_decay)
n_hidden_state = 16
divide_value_by = 20.0

n_actions = int(env.action_space.n)

h = DummyRepresentationModel(dim_obs_in=env.observation_space.shape[0], n_hidden=16, n_hidden_state=n_hidden_state)
f = DummyPredictionModel(n_hidden_state, n_actions=n_actions)
g = DummyDynamicsModel(n_hidden_state=n_hidden_state, n_actions=n_actions)

optimizer = torch.optim.Adam(params=list(h.parameters()) + list(f.parameters()) + list(g.parameters()), lr=alpha, weight_decay=c)

In [7]:
writer = SummaryWriter(log_dir=f"../runs/MuZero_{env_name}")
hparam_dict={
    'gamma' : gamma,
    'alpha' : alpha, # TODO this is not given in the paper -> look up in pseudocode
    'buffer_size' : buffer_size,
    'mini_batch_size' : mini_batch_size,
    'n_train_iterations' : n_train_iterations, # how many mini batches are sampled from the buffer
    'n_MCTS_sims_data_generation' : n_MCTS_sims_data_generation,
    'use_MCTS_in_evaluation' : use_MCTS_in_evaluation,
    'n_MCTS_sims_evaluation' : n_MCTS_sims_evaluation,
    'c' : c, # L2 penalty (used in Adam weight_decay)
    'n_hidden_state' : n_hidden_state
}
metric_dict={}
writer.add_hparams(hparam_dict=hparam_dict,metric_dict=metric_dict)

In [8]:
buffer_S          = torch.empty(size=(buffer_size, *env.observation_space.shape), dtype=torch.float32)
buffer_A          = torch.empty(size=(buffer_size, 1), dtype=torch.int32)
buffer_R          = torch.empty(size=(buffer_size, 1), dtype=torch.float32)
buffer_S_prime    = torch.empty(size=(buffer_size, *env.observation_space.shape), dtype=torch.float32)
buffer_terminated = torch.empty(size=(buffer_size, 1), dtype=torch.bool)
buffer_truncated  = torch.empty(size=(buffer_size, 1), dtype=torch.bool)
buffer_nu         = torch.empty(size=(buffer_size, 1), dtype=torch.float32)
buffer_pi         = torch.empty(size=(buffer_size, n_actions), dtype=torch.float32)

In [ ]:
# Generate experience with saved data_generation_model
# Refresh entire buffer

i_epoch = -1

# while True:
#     i_epoch +=1

h_checkpoint = copy.deepcopy(h)
f_checkpoint = copy.deepcopy(f)
g_checkpoint = copy.deepcopy(g)
for model in [h_checkpoint, f_checkpoint, g_checkpoint]:
    for param in model.parameters():
        param.requires_grad = False

S, info = env.reset()
done = False


# TODO this should be train=False, same for MCTS
for istep in range(buffer_size):

    # here, action selection is random sampling according to MCTS-improved policy + noise
    # but we don't do MCTS for now, so it's just sample + noise
    # torch.multinomial(pk, num_samples=1)
    with torch.no_grad():
        hidden_state                          = h_checkpoint(torch.tensor(S, dtype=torch.float32))
        policy_logits_pred, value_logits_pred = f_checkpoint(hidden_state)

        tree = MCTSTree(
            MCTSNode(
                hidden_state=hidden_state,
                n_actions=n_actions,
                p=F.softmax(policy_logits_pred, dim=-1),
                v = value_logits_pred,
                identifier="root",
                ),
            n_simulations=n_MCTS_sims_data_generation,
            dynamics_function=g_checkpoint,
            prediction_function=f_checkpoint,
            n_actions=n_actions
        )
        nu, pi = tree.search()

        A = tree.sample()

    S_prime, R, terminated, truncated, info = env.step(A.item())

    # Store in buffer
    # for now, save states, actions, rewards, next states, truncated/terminated
    # Calculate n-step returns later
    # Re-calculate or update save method for any values of the play_network
    buffer_S[istep, ...]          = torch.tensor(S, dtype=buffer_S.dtype)
    buffer_A[istep, ...]          = A.to(dtype=buffer_A.dtype)
    buffer_R[istep, ...]          = torch.tensor(R, dtype=buffer_R.dtype)
    buffer_S_prime[istep, ...]    = torch.tensor(S_prime, dtype=buffer_S_prime.dtype)
    buffer_terminated[istep, ...] = torch.tensor(terminated, dtype=buffer_terminated.dtype)
    buffer_truncated[istep, ...]  = torch.tensor(truncated, dtype=buffer_truncated.dtype)
    
    buffer_nu[istep, ...] = nu
    buffer_pi[istep, ...] = pi

    done = terminated or truncated

    if done:
        S, info = env.reset()
        done = False
    else:
        S = S_prime
    

    writer.add_scalar('MCTS/v',     value_logits_pred[0].detach().mean().item(),     i_epoch*buffer_size + istep)
    writer.add_scalar('MCTS/nu',     nu[0].detach().mean().item(),     i_epoch*buffer_size + istep)
    # TODO fix why do the dictionaries not show up in tensorboard
    writer.add_scalars('MCTS/p',     {'0': F.softmax(policy_logits_pred, dim=-1)[0].detach().item(), '1': F.softmax(policy_logits_pred, dim=-1)[1].detach().item()},     i_epoch*buffer_size + istep)
    writer.add_scalars('MCTS/pi',     {'0': pi[0].detach().item(), '1': pi[1].detach().item()},     i_epoch*buffer_size + istep)

In [42]:
# sample a mini batch, calc and train
# TODO prioritized sampling
# Reevaluated model: hidden_state, policy_logits_pred, value_logits_pred, reward_logits_pred, next_state_pred
# TODO this is pseudo code: this should be a sum over k, K successive steps, evaluations of the current model along 5 steps of generated data
# TODO rename R to u, and reward predictions to r
# TODO compute z for every state in the buffer, then compute p=abs(nu - z) for every state, then do prioritized sampling
 
K = 2

for i_iteration in range(n_train_iterations):
    ixs = np.random.default_rng().choice(buffer_size-K, mini_batch_size, replace=False) # -K+1 so that we don't access elements out of bounds # TODO for n=10 we need max(K, 10) successive entries from the buffer

    mb_S1 = buffer_S[ixs, ...]
    mb_A1 = buffer_A[ixs, ...]
    mb_R1 = buffer_R[ixs, ...]
    mb_S_prime1 = buffer_S_prime[ixs, ...]
    mb_terminated1 = buffer_terminated[ixs, ...]
    mb_truncated1 = buffer_truncated[ixs, ...]
    mb_nu2 = buffer_nu[ixs+1, ...] # for the next state
    mb_pi1 = buffer_pi[ixs, ...]

    mb_S2 = buffer_S[ixs+1, ...]
    mb_A2 = buffer_A[ixs+1, ...]
    mb_R2 = buffer_R[ixs+1, ...]
    mb_S_prime2 = buffer_S_prime[ixs+1, ...]
    mb_terminated2 = buffer_terminated[ixs+1, ...]
    mb_truncated2 = buffer_truncated[ixs+1, ...]
    mb_nu3 = buffer_nu[ixs+1+1, ...] # the next-next state
    mb_pi2 = buffer_pi[ixs+1, ...]


    # Treatment of terminal states as absorbing states
    # reward 0, value 0, uniform policy, action random (of all subsequent states TODO)
    mb_terminal1 = torch.logical_or(mb_terminated1, mb_truncated1)
    mb_R2 = torch.where(mb_terminal1, torch.zeros_like(mb_R2), mb_R2)
    mb_nu2 = torch.where(mb_terminal1, torch.zeros_like(mb_nu2), mb_nu2)
    mb_nu3 = torch.where(mb_terminal1, torch.zeros_like(mb_nu3), mb_nu3)
    mb_pi2 = torch.where(mb_terminal1, torch.ones(size=(mini_batch_size, n_actions))/n_actions, mb_pi2) 
    mb_A2 = torch.where(mb_terminal1, torch.multinomial(mb_pi2, num_samples=1), mb_A2)

    mb_terminal2 = torch.logical_or(mb_terminated2, mb_truncated2)
    mb_nu3 = torch.where(mb_terminal2, torch.zeros_like(mb_nu3), mb_nu3)

    mb_hidden_state1 = h(mb_S1)

    mb_p_pred1, mb_v_pred1 = f(mb_hidden_state1)
    mb_R_pred1, mb_hidden_state2 = g(mb_hidden_state1, mb_A1)

    mb_p_pred2, mb_v_pred2 = f(mb_hidden_state2)
    mb_R_pred2, mb_hidden_state3 = g(mb_hidden_state2, mb_A2)

    mb_z1 = mb_R1 + gamma * mb_nu2
    mb_z2 = mb_R2 + gamma * mb_nu3

    mb_loss1 = l_r(mb_R_pred1, mb_R1) + l_v(mb_v_pred1, mb_z1)/divide_value_by + l_p(mb_p_pred1, mb_pi1)
    mb_loss2 = l_r(mb_R_pred2, mb_R2) + l_v(mb_v_pred2, mb_z2)/divide_value_by + l_p(mb_p_pred2, mb_pi2)
    
    loss = (1.0/K)*(mb_loss1.mean() + mb_loss2.mean())

    # Train one step
    for model in [h, f, g]:
        for param in model.parameters():
            param.grad = None
    loss.backward()
    optimizer.step()

    writer.add_scalar('Loss/loss',   loss.detach().item(),                           i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Loss/mb_R1',  l_r(mb_R_pred1, mb_R1).mean().detach().item(),  i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Loss/mb_R2',  l_r(mb_R_pred2, mb_R2).mean().detach().item(),  i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Loss/mb_z1',  l_v(mb_v_pred1, mb_z1).mean().detach().item(),  i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Loss/mb_z2',  l_v(mb_v_pred2, mb_z2).mean().detach().item(),  i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Loss/mb_pi1', l_p(mb_p_pred1, mb_pi1).mean().detach().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Loss/mb_pi2', l_p(mb_p_pred2, mb_pi2).mean().detach().item(), i_epoch*n_train_iterations + i_iteration)

    writer.add_scalar('Values1/mb_R_pred1', mb_R_pred1[0].detach().mean().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values1/mb_R1',      mb_R1[0].detach().mean().item(),      i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values1/mb_v_pred1', mb_v_pred1[0].detach().mean().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values1/mb_z1',      mb_z1[0].detach().mean().item(),      i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values1/mb_p_pred1', mb_p_pred1[0, 0].detach().mean().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values1/mb_pi1',     mb_pi1[0, 0].detach().mean().item(),     i_epoch*n_train_iterations + i_iteration)
    
    writer.add_scalar('Values2/mb_R_pred2', mb_R_pred2[0].detach().mean().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values2/mb_R2',      mb_R2[0].detach().mean().item(),      i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values2/mb_v_pred2', mb_v_pred2[0].detach().mean().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values2/mb_z2',      mb_z2[0].detach().mean().item(),      i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values2/mb_p_pred2', mb_p_pred2[0, 0].detach().mean().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values2/mb_pi2',     mb_pi2[0, 0].detach().mean().item(),     i_epoch*n_train_iterations + i_iteration)

    # writer.add_scalar('MCTS/mb_nu2',     mb_nu2[0].detach().mean().item(),     i_epoch*n_train_iterations + i_iteration)
    # writer.add_scalar('MCTS/mb_nu3',     mb_nu2[0].detach().mean().item(),     i_epoch*n_train_iterations + i_iteration)



In [ ]:
# Evaluation

S, info = env.reset()
total_reward = 0
done = False

while not done:
    with torch.no_grad():
        hidden_state = h(torch.tensor(S, dtype=torch.float32))
        policy_logits_pred, value_logits_pred = f(hidden_state)
        
        if use_MCTS_in_evaluation:
            tree = MCTSTree(
                MCTSNode(
                    hidden_state=hidden_state,
                    n_actions=n_actions,
                    p=F.softmax(policy_logits_pred, dim=-1),
                    v = value_logits_pred,
                    identifier="root",
                    ),
                n_simulations=n_MCTS_sims_evaluation,
                dynamics_function=g,
                prediction_function=f,
                n_actions=n_actions
            )
            nu, pi = tree.search()
            A = tree.sample()
        else:
            A = torch.multinomial(F.softmax(policy_logits_pred, dim=-1), 1) # TODO probability tensor contains either `inf`, `nan` or element < 0
    
    S, R, terminated, truncated, info = env.step(A.item())
    total_reward += R
    done = terminated or truncated

print(total_reward)
writer.add_scalar('Eval/Reward', total_reward, i_epoch)


In [ ]:
#
writer.close()